# plan_quiz — research notebook

The **fan-in node**: merges the two parallel branches (concepts from `analyze_page`, `learner_context` from `load_memory`) into a blueprint that `generate_questions` executes one slot at a time.

Pure Python, no LLM — the judging already happened upstream; planning is applying rules:

1. **Count rule:** `final = min(requested, HARD_CAP 20, max_supportable)`
2. **Personalization:** weak concepts first, new concepts next, mastered last (bumped one Bloom level up)
3. **Variants before shrinking:** if concepts run out, reuse the most salient ones at a different Bloom level — every slot stays a unique (concept, bloom) pair
4. Flag the user only when the delivered count falls notably short of the request

**Resilience rule: never raises.**

In [ ]:
from typing import TypedDict, Literal

BloomLevel = Literal["remember", "understand", "apply", "analyse", "evaluate", "create"]
BLOOM_LADDER: list[BloomLevel] = ["remember", "understand", "apply", "analyse", "evaluate", "create"]

class Concept(TypedDict):
    name: str
    salience: float
    supporting_span: str

class LearnerContext(TypedDict):
    mastered_concepts: list[str]
    weak_concepts: list[str]
    recent_question_hashes: list[str]

class Slot(TypedDict):
    slot_id: int
    concept: str
    bloom_level: BloomLevel
    is_variant: bool         # True when the concept is reused at another level

class PlanResult(TypedDict):
    ok: bool
    blueprint: list[Slot]
    note: str                # user-facing, "" unless the gap is notable
    error: str

HARD_CAP = 20
NOTABLE_GAP = 0.7            # flag when delivered < 70% of requested
BASE_BLOOM: dict[str, BloomLevel] = {
    "easy": "understand", "medium": "apply", "hard": "analyse", "expert": "evaluate",
}

## The node

Slot assignment walks the ranked concepts round-robin: first pass at each concept's base level, later passes step up (then down) the Bloom ladder for variants.

In [ ]:
def _neighbors(base: BloomLevel) -> list[BloomLevel]:
    """Variant levels for a concept, nearest-first: up the ladder, then down."""
    i = BLOOM_LADDER.index(base)
    ups = BLOOM_LADDER[i+1:]
    downs = list(reversed(BLOOM_LADDER[:i]))
    order = [base]
    for k in range(max(len(ups), len(downs))):
        if k < len(ups): order.append(ups[k])
        if k < len(downs): order.append(downs[k])
    return order

def plan_quiz(concepts: list[Concept], ctx: LearnerContext, requested: int,
              difficulty: str, max_supportable: int) -> PlanResult:
    """Never raises: any failure degrades to an empty plan with an error."""
    try:
        final = max(0, min(int(requested), HARD_CAP, int(max_supportable)))
        if final == 0 or not concepts:
            return PlanResult(ok=False, blueprint=[], note="",
                              error="Page does not support any quiz questions.")

        base = BASE_BLOOM.get(difficulty, "apply")

        def bucket(c: Concept) -> int:
            if c["name"] in ctx["weak_concepts"]: return 0      # retest what's shaky
            if c["name"] not in ctx["mastered_concepts"]: return 1  # new material
            return 2                                             # mastered: last resort
        ranked = sorted(concepts, key=lambda c: (bucket(c), -c["salience"]))

        blueprint: list[Slot] = []
        used: set[tuple[str, str]] = set()
        pass_no = 0
        while len(blueprint) < final and pass_no < len(BLOOM_LADDER):
            for c in ranked:
                if len(blueprint) >= final:
                    break
                levels = _neighbors(base)
                if bucket(c) == 2:               # mastered: start one level harder
                    levels = _neighbors(base) [1:] or levels
                if pass_no >= len(levels):
                    continue
                key = (c["name"], levels[pass_no])
                if key in used:
                    continue
                used.add(key)
                blueprint.append(Slot(slot_id=len(blueprint), concept=c["name"],
                                      bloom_level=levels[pass_no], is_variant=pass_no > 0))
            pass_no += 1

        note = ""
        if len(blueprint) < requested * NOTABLE_GAP:
            note = (f"This page had enough substance for {len(blueprint)} good "
                    f"questions (you asked for {requested}).")
        return PlanResult(ok=True, blueprint=blueprint, note=note, error="")
    except Exception as e:                       # noqa: BLE001 — resilience boundary
        return PlanResult(ok=False, blueprint=[], note="", error=f"Planning failed: {e}")

## Try it

In [ ]:
concepts: list[Concept] = [
    {"name": "self-attention",       "salience": 0.95, "supporting_span": "..."},
    {"name": "multi-head attention", "salience": 0.8,  "supporting_span": "..."},
    {"name": "positional encoding",  "salience": 0.6,  "supporting_span": "..."},
]
ctx: LearnerContext = {"mastered_concepts": ["self-attention"],
                       "weak_concepts": ["multi-head attention"],
                       "recent_question_hashes": []}

# rich request: 6 questions from 3 concepts -> variants kick in
plan = plan_quiz(concepts, ctx, requested=6, difficulty="hard", max_supportable=10)
for s in plan["blueprint"]:
    print(s["slot_id"], s["concept"].ljust(22), s["bloom_level"].ljust(10), "variant" if s["is_variant"] else "")
print("note:", repr(plan["note"]))

In [ ]:
# checks: weak first, mastered bumped harder, all slots unique, count rule holds
b = plan["blueprint"]
assert b[0]["concept"] == "multi-head attention"                   # weak first
assert len({(s["concept"], s["bloom_level"]) for s in b}) == len(b)  # unique pairs
assert len(b) == 6

sa = [s for s in b if s["concept"] == "self-attention"]            # mastered
assert all(s["bloom_level"] != "analyse" or s["is_variant"] for s in sa)

shrunk = plan_quiz(concepts, ctx, requested=20, difficulty="easy", max_supportable=4)
assert len(shrunk["blueprint"]) == 4 and shrunk["note"]            # notable gap -> note
print("note on shrink:", shrunk["note"])

assert plan_quiz([], ctx, 5, "hard", 5)["ok"] is False             # graceful empty
assert plan_quiz(concepts, ctx, 5, "hard", "oops")["ok"] is False  # graceful bad input
print("all checks passed")

## Notes

- **Fan-in, no reducers:** the two branches wrote different state keys; this node just reads both.
- Blueprint slots carry everything `generate_questions` needs; it generates one question per slot and `repair_questions` regenerates specific slots by `slot_id`.
- Mastered concepts start one Bloom level above the profile base — retesting what someone knows, but harder.
- `note` flows through state to `finalize_quiz` and out via SSE, only when the gap is notable.
- Tune later: NOTABLE_GAP, and whether `recent_question_hashes` should exclude a (concept, bloom) pair here rather than at generation time.